# Day 4 -- Fund Performance Analytics
## Bluestock MF Capstone

**Deliverables:** fund_scorecard.csv, alpha_beta.csv, benchmark_comparison.png

**Tasks:** Daily returns, CAGR 1Y/3Y/5Y, Sharpe, Sortino, Alpha/Beta (OLS vs Nifty 100), Max Drawdown, Fund Scorecard (0-100), Benchmark comparison chart

In [1]:
from pathlib import Path
import pandas as pd, numpy as np, sqlite3
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats as sp_stats
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
plt.rcParams.update({'figure.dpi':130, 'axes.spines.top':False, 'axes.spines.right':False})

BASE   = Path().resolve().parent
DB     = BASE / "data" / "db" / "bluestock_mf.db"
PROC   = BASE / "data" / "processed"
CHARTS = BASE / "data" / "charts"
CHARTS.mkdir(exist_ok=True)

RISK_FREE    = 0.065
TRADING_DAYS = 252
rf_daily     = RISK_FREE / TRADING_DAYS

with sqlite3.connect(DB) as conn:
    nav      = pd.read_sql_query("SELECT scheme_code, date_key, nav, daily_return FROM fact_nav ORDER BY scheme_code, date_key", conn)
    fund     = pd.read_sql_query("SELECT scheme_code, scheme_name, sub_category, category, amc_name, benchmark, risk_level, expense_ratio FROM dim_fund", conn)
    bm       = pd.read_sql_query("SELECT date, index_name, close_value, daily_return FROM ref_benchmark_indices ORDER BY index_name, date", conn)

nav["date"] = pd.to_datetime(nav["date_key"])
bm["date"]  = pd.to_datetime(bm["date"])
print(f"nav={len(nav):,} rows | funds={nav.scheme_code.nunique()} | bm={bm.index_name.nunique()} indices")


nav=46,000 rows | funds=40 | bm=7 indices


In [2]:
# Task 1: Daily Returns Distribution
returns_all = nav['daily_return'].dropna()
stats = {
    'Mean %':     round(returns_all.mean()*100, 4),
    'Std Dev %':  round(returns_all.std()*100, 4),
    'Min %':      round(returns_all.min()*100, 4),
    'Max %':      round(returns_all.max()*100, 4),
    'Skewness':   round(returns_all.skew(), 4),
    'Kurtosis':   round(returns_all.kurtosis(), 4),
    'Outliers >10%': int((returns_all.abs()>0.10).sum()),
}
print("DAILY RETURN STATS:")
for k,v in stats.items(): print(f"  {k:20s}: {v}")

fig, axes = plt.subplots(1,2,figsize=(13,4))
axes[0].hist(returns_all.clip(-0.06,0.06), bins=100, color='steelblue', edgecolor='none', alpha=0.8)
axes[0].axvline(0, color='black', lw=1, ls='--')
axes[0].set_title('Daily Return Distribution (All Funds, clipped +-6%)')
axes[0].set_xlabel('Daily Return'); axes[0].set_ylabel('Frequency')

per_fund = (nav.groupby('scheme_code')['daily_return'].mean().reset_index()
            .merge(fund[['scheme_code','scheme_name']], on='scheme_code')
            .sort_values('daily_return', ascending=True))
colors_ret = ['#e74c3c' if v < 0 else '#27ae60' for v in per_fund['daily_return']]
axes[1].barh([n[:22] for n in per_fund['scheme_name']], per_fund['daily_return']*100,
             color=colors_ret, edgecolor='none', height=0.65)
axes[1].axvline(0, color='black', lw=0.8)
axes[1].set_title('Avg Daily Return per Fund (%)'); axes[1].tick_params(axis='y', labelsize=7)
plt.tight_layout()
plt.savefig(CHARTS/'chart_d4_daily_returns.png', bbox_inches='tight')
plt.show()
print("Saved: chart_d4_daily_returns.png")


DAILY RETURN STATS:
  Mean %              : 0.0631
  Std Dev %           : 1.029
  Min %               : -5.8102
  Max %               : 6.4713
  Skewness            : 0.0427
  Kurtosis            : 1.3673
  Outliers >10%       : 0
Saved: chart_d4_daily_returns.png


In [3]:
# Task 2: CAGR 1Y / 3Y / 5Y / Inception
def compute_cagr(nav_series, years):
    if len(nav_series) < 2: return np.nan
    start, end = nav_series.iloc[0], nav_series.iloc[-1]
    return (end/start)**(1.0/years) - 1 if start > 0 else np.nan

cagr_records = []
for code, grp in nav.groupby('scheme_code'):
    grp  = grp.sort_values('date')
    row  = {'scheme_code': code}
    for label, n_days in [('1Y',252),('3Y',756),('5Y',1260)]:
        w = grp.tail(n_days)
        row[f'cagr_{label}'] = compute_cagr(w['nav'], n_days/TRADING_DAYS) * 100
    row['cagr_inception'] = compute_cagr(grp['nav'], len(grp)/TRADING_DAYS) * 100
    cagr_records.append(row)

cagr_df = (pd.DataFrame(cagr_records)
           .merge(fund[['scheme_code','scheme_name','sub_category','category','amc_name','risk_level']], on='scheme_code')
           .sort_values('cagr_3Y', ascending=False).reset_index(drop=True))

display = cagr_df[['scheme_name','sub_category','cagr_1Y','cagr_3Y','cagr_5Y','cagr_inception']].copy()
display.columns = ['Fund','Sub-Category','1Y CAGR%','3Y CAGR%','5Y CAGR%','Inception%']
for c in ['1Y CAGR%','3Y CAGR%','5Y CAGR%','Inception%']:
    display[c] = display[c].round(2)
print("CAGR TABLE (sorted by 3Y CAGR):")
print(display.to_string(index=False))


CAGR TABLE (sorted by 3Y CAGR):
                                                 Fund    Sub-Category  1Y CAGR%  3Y CAGR%  5Y CAGR%  Inception%
                  Axis Midcap Fund - Regular - Growth         Mid Cap     30.92     36.07     24.45       27.08
   HDFC Mid-Cap Opportunities Fund - Regular - Growth         Mid Cap     47.73     33.63     26.07       28.90
        ABSL Frontline Equity Fund - Regular - Growth       Large Cap     45.09     32.53     20.44       22.60
        Mirae Asset Large Cap Fund - Regular - Growth       Large Cap     14.58     31.28     26.80       29.71
             ICICI Pru Midcap Fund - Regular - Growth         Mid Cap     30.35     30.21     28.38       31.48
            ICICI Pru Bluechip Fund - Direct - Growth       Large Cap     11.50     28.37     20.23       22.37
           SBI Small Cap Fund - Regular Plan - Growth       Small Cap     84.53     27.81     28.03       31.10
            SBI Bluechip Fund - Regular Plan - Growth       Large Cap   

In [4]:
# Task 3: Sharpe Ratio
sharpe_records = []
for code, grp in nav.groupby('scheme_code'):
    rets = grp['daily_return'].dropna()
    if len(rets) < 60: continue
    ann_ret = (1 + rets.mean())**TRADING_DAYS - 1
    ann_std = rets.std() * np.sqrt(TRADING_DAYS)
    sharpe  = (ann_ret - RISK_FREE) / ann_std if ann_std > 0 else np.nan
    name    = fund.loc[fund['scheme_code']==code,'scheme_name'].values[0]
    sharpe_records.append({'scheme_code':code,'scheme_name':name,
                           'ann_return':ann_ret*100,'ann_std':ann_std*100,'sharpe_ratio':sharpe})

sharpe_df = pd.DataFrame(sharpe_records).sort_values('sharpe_ratio', ascending=False)
print("SHARPE RATIO RANKING (Rf = 6.5%)")
print(f"{'Fund':40s} {'AnnRet%':>9} {'AnnVol%':>9} {'Sharpe':>9}")
print("-"*70)
for _, r in sharpe_df.iterrows():
    flag = " *" if r['sharpe_ratio'] >= 1 else ""
    print(f"{r['scheme_name'][:40]:40s} {r['ann_return']:>9.2f} {r['ann_std']:>9.2f} {r['sharpe_ratio']:>9.3f}{flag}")
print(f"\n* Sharpe >= 1.0: {(sharpe_df['sharpe_ratio']>=1.0).sum()} funds")


SHARPE RATIO RANKING (Rf = 6.5%)
Fund                                       AnnRet%   AnnVol%    Sharpe
----------------------------------------------------------------------
Mirae Asset Large Cap Fund - Regular - G     31.05     14.19     1.730 *
Kotak Flexicap Fund - Regular - Growth       31.32     15.89     1.562 *
Mirae Asset Tax Saver Fund - Regular - G     32.72     17.67     1.484 *
ICICI Pru Midcap Fund - Regular - Growth     33.97     19.29     1.424 *
SBI Bluechip Fund - Regular Plan - Growt     25.98     13.74     1.417 *
DSP Midcap Fund - Regular - Growth           30.44     17.75     1.349 *
HDFC Mid-Cap Opportunities Fund - Regula     31.25     18.94     1.307 *
Nippon India Large Cap Fund - Regular -      24.35     14.15     1.262 *
ABSL Frontline Equity Fund - Regular - G     23.93     14.57     1.197 *
ICICI Pru Bluechip Fund - Direct - Growt     23.66     14.36     1.195 *
Axis Midcap Fund - Regular - Growth          29.51     19.41     1.186 *
DSP Small Cap Fund - R

In [5]:
# Task 4: Sortino Ratio
sortino_records = []
for code, grp in nav.groupby('scheme_code'):
    rets = grp['daily_return'].dropna()
    if len(rets) < 60: continue
    ann_ret  = (1 + rets.mean())**TRADING_DAYS - 1
    downside = rets[rets < 0]
    down_std = downside.std() * np.sqrt(TRADING_DAYS) if len(downside) > 5 else np.nan
    sortino  = (ann_ret - RISK_FREE) / down_std if (down_std and down_std > 0) else np.nan
    name     = fund.loc[fund['scheme_code']==code,'scheme_name'].values[0]
    sortino_records.append({'scheme_code':code,'scheme_name':name,
                            'ann_return':ann_ret*100,'downside_std':down_std*100 if down_std else np.nan,
                            'sortino_ratio':sortino})
sortino_df = pd.DataFrame(sortino_records).sort_values('sortino_ratio', ascending=False)

# Sharpe vs Sortino chart
merged = sharpe_df[['scheme_code','scheme_name','sharpe_ratio']].merge(
         sortino_df[['scheme_code','sortino_ratio']], on='scheme_code')
merged['short_name'] = [n[:22] for n in merged['scheme_name']]
merged = merged.sort_values('sharpe_ratio', ascending=True)

fig, ax = plt.subplots(figsize=(12,9))
y = np.arange(len(merged))
ax.barh(y-0.2, merged['sharpe_ratio'],  0.38, label='Sharpe',  color='#3498db', alpha=0.85)
ax.barh(y+0.2, merged['sortino_ratio'], 0.38, label='Sortino', color='#e67e22', alpha=0.85)
ax.set_yticks(y); ax.set_yticklabels(merged['short_name'], fontsize=8)
ax.axvline(0, color='black', lw=0.8)
ax.axvline(1, color='green', lw=1, ls='--', alpha=0.6, label='Threshold=1')
ax.set_title('Sharpe vs Sortino Ratio (Rf=6.5%)', fontweight='bold')
ax.set_xlabel('Ratio'); ax.legend()
plt.tight_layout()
plt.savefig(CHARTS/'chart_d4_sharpe_sortino.png', bbox_inches='tight')
plt.show()
print("Saved: chart_d4_sharpe_sortino.png")


Saved: chart_d4_sharpe_sortino.png


In [6]:
# Task 5: Alpha & Beta vs Nifty 100
nifty100_ret = bm[bm['index_name']=='Nifty 100'].set_index('date')['daily_return'].dropna()
ab_records   = []

for code, grp in nav.groupby('scheme_code'):
    rets    = grp.set_index('date')['daily_return'].dropna()
    aligned = pd.concat([rets, nifty100_ret], axis=1, join='inner').dropna()
    aligned.columns = ['fund','bench']
    if len(aligned) < 60: continue
    f_ex = aligned['fund']  - rf_daily
    b_ex = aligned['bench'] - rf_daily
    slope, intercept, r_val, _, _ = sp_stats.linregress(b_ex, f_ex)
    te   = (aligned['fund'] - aligned['bench']).std() * np.sqrt(TRADING_DAYS)
    name = fund.loc[fund['scheme_code']==code,'scheme_name'].values[0]
    sub  = fund.loc[fund['scheme_code']==code,'sub_category'].values[0]
    ab_records.append({'scheme_code':code,'scheme_name':name,'sub_category':sub,
                       'alpha_ann':round(intercept*TRADING_DAYS*100,4),
                       'beta':round(slope,4),'r_squared':round(r_val**2,4),
                       'tracking_error':round(te*100,4),'n_obs':len(aligned)})

ab_df = pd.DataFrame(ab_records).sort_values('alpha_ann', ascending=False)
ab_df.to_csv(PROC/'alpha_beta.csv', index=False)
print(f"Saved: alpha_beta.csv ({len(ab_df)} rows)")
print()
print(f"{'Fund':38s} {'Alpha%':>8} {'Beta':>7} {'R2':>7} {'TE%':>7}")
print("-"*65)
for _, r in ab_df.iterrows():
    print(f"{r['scheme_name'][:38]:38s} {r['alpha_ann']:>8.2f} {r['beta']:>7.3f} {r['r_squared']:>7.3f} {r['tracking_error']:>7.2f}")


KeyError: 'alpha_ann'

In [7]:
# Alpha & Beta Charts
fig, axes = plt.subplots(1,2,figsize=(14,6))
ab_s = ab_df.sort_values('alpha_ann', ascending=True)
colors_a = ['#e74c3c' if v < 0 else '#27ae60' for v in ab_s['alpha_ann']]
axes[0].barh([n[:25] for n in ab_s['scheme_name']], ab_s['alpha_ann'],
             color=colors_a, edgecolor='none', height=0.65)
axes[0].axvline(0, color='black', lw=1)
axes[0].set_title("Jensen's Alpha % (vs Nifty 100)", fontweight='bold')
axes[0].tick_params(axis='y', labelsize=7.5)

for _, r in ab_df.iterrows():
    axes[1].scatter(r['beta'], r['alpha_ann'], s=70, edgecolors='white', lw=0.5, zorder=3)
    axes[1].annotate(r['scheme_name'][:12], (r['beta'], r['alpha_ann']),
                     fontsize=6, alpha=0.7, xytext=(4,3), textcoords='offset points')
axes[1].axhline(0, color='black', lw=0.7, ls='--')
axes[1].axvline(1, color='red', lw=0.8, ls='--', alpha=0.5, label='Beta=1')
axes[1].set_title('Alpha vs Beta (vs Nifty 100)', fontweight='bold')
axes[1].set_xlabel('Beta'); axes[1].set_ylabel('Alpha %'); axes[1].legend()
plt.tight_layout()
plt.savefig(CHARTS/'chart_d4_alpha_beta.png', bbox_inches='tight')
plt.show()


NameError: name 'ab_df' is not defined

In [8]:
# Task 6: Maximum Drawdown
mdd_records = []
for code, grp in nav.groupby('scheme_code'):
    grp  = grp.sort_values('date')
    peak = grp['nav'].cummax()
    dd   = (grp['nav'] - peak) / peak
    mdd  = dd.min()
    trough_date = grp.loc[dd.idxmin(),'date']
    peak_date   = grp.loc[:dd.idxmin(),'nav'].idxmax()
    peak_date   = grp.loc[peak_date,'date']
    name = fund.loc[fund['scheme_code']==code,'scheme_name'].values[0]
    sub  = fund.loc[fund['scheme_code']==code,'sub_category'].values[0]
    mdd_records.append({'scheme_code':code,'scheme_name':name,'sub_category':sub,
                        'max_drawdown_pct':round(mdd*100,2),
                        'peak_date':str(peak_date.date()),
                        'trough_date':str(trough_date.date()),
                        'duration_days':(trough_date-peak_date).days})

mdd_df = pd.DataFrame(mdd_records).sort_values('max_drawdown_pct')
print("MAX DRAWDOWN (worst to least)")
print(f"{'Fund':35s} {'MDD%':>7} {'Peak':>12} {'Trough':>12} {'Duration':>10}")
print("-"*80)
for _, r in mdd_df.iterrows():
    print(f"{r['scheme_name'][:35]:35s} {r['max_drawdown_pct']:>7.2f} {r['peak_date']:>12} {r['trough_date']:>12} {r['duration_days']:>9}d")


MAX DRAWDOWN (worst to least)
Fund                                   MDD%         Peak       Trough   Duration
--------------------------------------------------------------------------------
SBI Small Cap Fund - Direct Plan -   -52.57   2023-01-17   2025-10-28      1015d
Axis Small Cap Fund - Regular - Gro  -51.68   2025-05-22   2026-05-11       354d
ABSL Small Cap Fund - Regular - Gro  -35.45   2024-11-21   2026-05-11       536d
DSP Small Cap Fund - Regular - Grow  -31.17   2024-05-03   2025-01-03       245d
SBI Small Cap Fund - Regular Plan -  -28.71   2024-08-28   2025-05-14       259d
UTI Mid Cap Fund - Regular - Growth  -28.00   2025-01-07   2026-04-27       475d
HDFC Top 100 Fund - Regular Plan -   -24.73   2022-03-30   2022-09-15       169d
Kotak Emerging Equity Fund - Regula  -24.00   2023-11-09   2024-10-17       343d
Nippon India Small Cap Fund - Regul  -23.34   2025-04-09   2026-02-20       317d
Axis Bluechip Fund - Direct - Growt  -21.75   2022-02-24   2023-05-22       452

In [9]:
# Task 7: Fund Scorecard
# Guard: ensure ab_df exists (may need rerun if previous cell errored)
try:
    _ = ab_df
except NameError:
    from scipy import stats as sp_stats
    rf_daily = RISK_FREE / TRADING_DAYS
    nifty100_ret = bm[bm['index_name']=='Nifty 100'].set_index('date')['daily_return'].dropna()
    ab_records = []
    for code, grp in nav.groupby('scheme_code'):
        rets = grp.set_index('date')['daily_return'].dropna()
        aligned = pd.concat([rets, nifty100_ret], axis=1, join='inner').dropna()
        aligned.columns = ['fund','bench']
        if len(aligned) < 60: continue
        f_ex = aligned['fund'] - rf_daily; b_ex = aligned['bench'] - rf_daily
        slope, intercept, r_val, _, _ = sp_stats.linregress(b_ex, f_ex)
        te   = (aligned['fund'] - aligned['bench']).std() * np.sqrt(TRADING_DAYS)
        name = fund.loc[fund['scheme_code']==code,'scheme_name'].values[0]
        sub  = fund.loc[fund['scheme_code']==code,'sub_category'].values[0]
        ab_records.append({'scheme_code':code,'scheme_name':name,'sub_category':sub,
                           'alpha_ann':round(intercept*TRADING_DAYS*100,4),
                           'beta':round(slope,4),'r_squared':round(r_val**2,4),
                           'tracking_error':round(te*100,4),'n_obs':len(aligned)})
    ab_df = pd.DataFrame(ab_records).sort_values('alpha_ann', ascending=False)
    ab_df.to_csv(PROC/'alpha_beta.csv', index=False)
    print("Recomputed ab_df")

scorecard = (cagr_df.merge(sharpe_df[['scheme_code','sharpe_ratio']], on='scheme_code', how='left')
             .merge(sortino_df[['scheme_code','sortino_ratio']], on='scheme_code', how='left')
             .merge(ab_df[['scheme_code','alpha_ann','beta','r_squared','tracking_error']], on='scheme_code', how='left')
             .merge(mdd_df[['scheme_code','max_drawdown_pct']], on='scheme_code', how='left')
             .merge(fund[['scheme_code','expense_ratio']], on='scheme_code', how='left'))

def rank_pct(series, asc=True):
    r = series.rank(ascending=asc, na_option='bottom')
    return (r / r.max() * 100).round(2)

scorecard['r_3yr']  = rank_pct(scorecard['cagr_3Y'],      asc=False)
scorecard['r_sh']   = rank_pct(scorecard['sharpe_ratio'],  asc=False)
scorecard['r_al']   = rank_pct(scorecard['alpha_ann'],     asc=False)
scorecard['r_er']   = rank_pct(scorecard['expense_ratio'], asc=True)
scorecard['r_mdd']  = rank_pct(scorecard['max_drawdown_pct'], asc=True)
scorecard['composite_score'] = (0.30*scorecard['r_3yr'] + 0.25*scorecard['r_sh'] +
                                0.20*scorecard['r_al']  + 0.15*scorecard['r_er'] +
                                0.10*scorecard['r_mdd']).round(2)
scorecard = scorecard.sort_values('composite_score', ascending=False).reset_index(drop=True)
scorecard.index += 1

save_cols = ['scheme_code','scheme_name','sub_category','category','amc_name','risk_level',
             'cagr_1Y','cagr_3Y','cagr_5Y','cagr_inception','sharpe_ratio','sortino_ratio',
             'alpha_ann','beta','r_squared','max_drawdown_pct','expense_ratio','composite_score']
scorecard[save_cols].to_csv(PROC/'fund_scorecard.csv', index=True, index_label='rank')
print(f"Saved: fund_scorecard.csv ({len(scorecard)} rows)")
print()
print("TOP 20 FUND SCORECARD:")
print(f"{'Rank':>4}  {'Fund':38s} {'Score':>7} {'CAGR3Y':>8} {'Sharpe':>8}")
print("-"*75)
for rank, r in scorecard.head(20).iterrows():
    print(f"{rank:>4}  {r['scheme_name'][:38]:38s} {r['composite_score']:>7.1f} {r['cagr_3Y']:>8.2f} {r['sharpe_ratio']:>8.3f}")


KeyError: 'alpha_ann'

In [10]:
# Scorecard chart
top20 = scorecard.head(20)
risk_pal = {'Low':'#27ae60','Moderate':'#f1c40f','Moderately High':'#e67e22','High':'#e74c3c'}
colors_sc = [risk_pal.get(r,'#3498db') for r in top20['risk_level']]
fig, ax = plt.subplots(figsize=(12,9))
bars = ax.barh(range(len(top20)), top20['composite_score'], color=colors_sc, edgecolor='none', height=0.65)
ax.set_yticks(range(len(top20)))
ax.set_yticklabels([f"{i+1}. {n[:30]}" for i,n in enumerate(top20['scheme_name'])], fontsize=8.5)
for bar, val in zip(bars, top20['composite_score']):
    ax.text(val+0.3, bar.get_y()+bar.get_height()/2, f"{val:.1f}", va='center', fontsize=8)
for lbl, clr in risk_pal.items():
    ax.scatter([],[],c=clr,label=lbl,s=50)
ax.legend(title='Risk', fontsize=8, loc='lower right')
ax.set_title('Fund Scorecard Top 20 (30% 3Y CAGR + 25% Sharpe + 20% Alpha + 15% ER + 10% MDD)', fontweight='bold')
ax.set_xlabel('Composite Score'); ax.invert_yaxis()
plt.tight_layout()
plt.savefig(CHARTS/'chart_d4_fund_scorecard.png', bbox_inches='tight')
plt.show()
print("Saved: chart_d4_fund_scorecard.png")


NameError: name 'scorecard' is not defined

In [11]:
# Task 8: Benchmark Comparison -- Top 5 vs Nifty 50 & Nifty 100
top5_codes = scorecard.head(5)['scheme_code'].values
start_3y   = nav['date'].max() - pd.DateOffset(years=3)

nifty50_s  = bm[bm['index_name']=='Nifty 50'].set_index('date')['close_value']
nifty100_s = bm[bm['index_name']=='Nifty 100'].set_index('date')['close_value']

fig, ax = plt.subplots(figsize=(14,6))
for bench_s, bench_name, ls, clr in [(nifty50_s,'Nifty 50','--','#2c3e50'),(nifty100_s,'Nifty 100','-.','#7f8c8d')]:
    b3y = bench_s[bench_s.index >= start_3y]
    if not b3y.empty:
        ax.plot(b3y.index, b3y/b3y.iloc[0]*100, lw=2.5, ls=ls, color=clr, label=bench_name, zorder=4)

colors_top5 = ['#e74c3c','#3498db','#27ae60','#e67e22','#9b59b6']
for code, clr in zip(top5_codes, colors_top5):
    f3y = nav[(nav['scheme_code']==code)&(nav['date']>=start_3y)].sort_values('date')
    if f3y.empty: continue
    idx  = f3y['nav']/f3y['nav'].iloc[0]*100
    name = fund.loc[fund['scheme_code']==code,'scheme_name'].values[0]
    # tracking error vs Nifty 100
    fr3y = f3y.set_index('date')['daily_return'].dropna()
    n100 = bm[(bm['index_name']=='Nifty 100')&(bm['date']>=start_3y)].set_index('date')['daily_return']
    al   = pd.concat([fr3y,n100],axis=1,join='inner').dropna()
    te   = (al.iloc[:,0]-al.iloc[:,1]).std()*np.sqrt(252)*100 if len(al)>10 else np.nan
    lbl  = f"{name[:25]} (TE:{te:.1f}%)" if not np.isnan(te) else name[:25]
    ax.plot(f3y['date'], idx, lw=2, color=clr, label=lbl, alpha=0.9)

ax.axhline(100, color='black', lw=0.7, ls=':', alpha=0.4)
ax.set_title('Top 5 Funds vs Nifty 50 & Nifty 100 -- 3-Year Indexed Performance', fontweight='bold')
ax.set_ylabel('Indexed (Base=100)'); ax.set_xlabel('Date')
ax.legend(fontsize=8, loc='upper left')
plt.tight_layout()
plt.savefig(CHARTS/'chart_benchmark_comparison.png', bbox_inches='tight')
plt.show()
print("Saved: chart_benchmark_comparison.png")


NameError: name 'scorecard' is not defined

In [12]:
# Final Deliverables Check
deliverables = [
    (PROC/'fund_scorecard.csv',             'fund_scorecard.csv'),
    (PROC/'alpha_beta.csv',                  'alpha_beta.csv'),
    (CHARTS/'chart_benchmark_comparison.png','benchmark_comparison.png'),
    (CHARTS/'chart_d4_daily_returns.png',    'daily_returns.png'),
    (CHARTS/'chart_d4_sharpe_sortino.png',   'sharpe_sortino.png'),
    (CHARTS/'chart_d4_alpha_beta.png',       'alpha_beta.png'),
    (CHARTS/'chart_d4_fund_scorecard.png',   'fund_scorecard.png'),
]
print("DAY 4 DELIVERABLES:")
for path, label in deliverables:
    ok   = "OK" if path.exists() else "MISSING"
    size = f"{path.stat().st_size//1024} KB" if path.exists() else ""
    print(f"  {ok}  {label:40s} {size}")
print()
print("Top 5 Composite Score:")
for rank, r in scorecard.head(5).iterrows():
    print(f"  {rank}. {r['scheme_name'][:40]:40s} Score:{r['composite_score']:.1f}")


DAY 4 DELIVERABLES:
  OK  fund_scorecard.csv                       9 KB
  OK  alpha_beta.csv                           3 KB
  OK  benchmark_comparison.png                 353 KB
  OK  daily_returns.png                        113 KB
  OK  sharpe_sortino.png                       152 KB
  OK  alpha_beta.png                           134 KB
  OK  fund_scorecard.png                       121 KB

Top 5 Composite Score:


NameError: name 'scorecard' is not defined